# 🔬 Microplastic Detection — YOLO26 Training (Kaggle · GPU P100)

End-to-end pipeline: **install → verify GPU → build data.yaml → train → validate → export (ONNX + PyTorch) → test inference → package for website**.

**Before running:** Notebook settings (right panel) →
- **Accelerator: GPU T4 x2**  ⭐ *(use T4, not P100 — Kaggle's current PyTorch 2.10+cu128 dropped Pascal/P100 support, so P100 errors with 'no kernel image'. T4 is Turing and fully supported.)*  Not TPU either — Ultralytics YOLO runs on PyTorch/CUDA, which TPU doesn't support.
- **Internet: ON**  (needed to pip-install ultralytics and download pretrained weights)
- **Add data:** attach the dataset you uploaded (folder with `train/ valid/ test/`).

Run all cells top to bottom. All downloadable outputs land in `/kaggle/working/`.

> ⚠️ **P100 note:** Kaggle's current image ships PyTorch 2.10+cu128, which has **no P100 (Pascal) kernels** — training on P100 fails with *'no kernel image is available'*. **Use GPU T4 x2.** (If you can only get a P100, replace cell 2's install with a `torch==2.5.1 ... cu121` downgrade + Kernel Restart — see the guide's troubleshooting table.)

## 1 · Environment & GPU check

In [ ]:
import subprocess, torch, platform
print('Python :', platform.python_version())
print('Torch  :', torch.__version__)
print('CUDA   :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU    :', torch.cuda.get_device_name(0))
    print('VRAM   : %.1f GB' % (torch.cuda.get_device_properties(0).total_memory/1e9))
else:
    print('⚠️  No GPU detected — set Accelerator = GPU P100 in notebook settings.')
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)

## 2 · Install Ultralytics (YOLO26) — **without breaking Kaggle's P100 PyTorch**
`pip install -U ultralytics` pulls a newer torch whose CUDA build has **no Pascal (sm_60) kernels**, which crashes the P100 with *'no kernel image is available for execution on the device'*. We upgrade **only ultralytics** (`--no-deps`) and keep Kaggle's working torch.

In [ ]:
import torch
print('Preinstalled torch:', torch.__version__, '| CUDA build:', torch.version.cuda)

# -U upgrades ultralytics to a YOLO26-capable version; --no-deps protects torch/torchvision.
!pip install -q -U ultralytics --no-deps
!pip install -q ultralytics-thop py-cpuinfo   # tiny pure-python deps, safe to add

import ultralytics, importlib
importlib.reload(ultralytics)
print('ultralytics', ultralytics.__version__)

# Smoke-test that THIS torch can actually run kernels on the P100 before training:
_x = torch.zeros(1, device='cuda') + 1
print('✅ CUDA kernel test passed on', torch.cuda.get_device_name(0))

## 3 · Locate the uploaded dataset (auto-detect)

In [ ]:
import glob, os
# Find the folder that contains train/images anywhere under /kaggle/input.
# This is robust to however the dataset zip was extracted.
cands = glob.glob('/kaggle/input/**/train/images', recursive=True)
assert cands, 'Could not find */train/images under /kaggle/input — did you attach the dataset?'
DATA_ROOT = os.path.dirname(os.path.dirname(cands[0]))
print('Dataset root:', DATA_ROOT)
for sp in ['train', 'valid', 'test']:
    ip = os.path.join(DATA_ROOT, sp, 'images')
    n = len(glob.glob(os.path.join(ip, '*'))) if os.path.isdir(ip) else 0
    print(f'  {sp:5s}: {n} images')

## 4 · Write the Kaggle `data.yaml`
The uploaded `data.yaml` points at a Windows path, so we regenerate it with the Kaggle mount path.

In [ ]:
data_yaml_path = '/kaggle/working/data.yaml'
yaml_text = f'''# Microplastic YOLO26 — Kaggle runtime config
path: {DATA_ROOT}
train: train/images
val: valid/images
test: test/images

nc: 4
names:
  0: fiber
  1: film
  2: fragment
  3: pellet
'''
with open(data_yaml_path, 'w') as f:
    f.write(yaml_text)
print(yaml_text)

## 5 · Training configuration
Tuned for a single **P100 (16 GB)** within Kaggle's 12 h GPU session.
Switch `MODEL` to `yolo26m.pt` for higher accuracy (slower), or `yolo26n.pt` for a smaller/faster web model.

In [ ]:
MODEL     = 'yolo26s.pt'   # n < s < m < l < x   (accuracy up, speed down)
EPOCHS    = 150
IMGSZ     = 640            # dataset has 480px & 640px imgs; 640 letterboxes cleanly
BATCH     = 16             # P100 16GB safe; try 32 if no OOM
PATIENCE  = 40             # early-stop if val mAP plateaus
WORKERS   = 4              # Kaggle gives ~4 vCPUs
SEED      = 42
RESUME    = False          # set True to continue a previous run (see notes at bottom)
PROJECT   = '/kaggle/working/microplastic'
RUN_NAME  = 'yolo26_run'

## 6 · Train

In [ ]:
from ultralytics import YOLO
import os

last_ckpt = os.path.join(PROJECT, RUN_NAME, 'weights', 'last.pt')
if RESUME and os.path.exists(last_ckpt):
    print('Resuming from', last_ckpt)
    model = YOLO(last_ckpt)
    resume_flag = True
else:
    try:
        model = YOLO(MODEL)                 # auto-downloads pretrained YOLO26 weights
    except Exception as e:
        print('YOLO26 weights unavailable (', e, ') — falling back to yolo11s.pt')
        model = YOLO('yolo11s.pt')
    resume_flag = False

results = model.train(
    data=data_yaml_path,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    patience=PATIENCE,
    workers=WORKERS,
    seed=SEED,
    device=0,
    project=PROJECT,
    name=RUN_NAME,
    exist_ok=True,
    resume=resume_flag,
    cache=False,           # 9k imgs — avoid RAM cache OOM on Kaggle
    amp=True,              # mixed precision — faster on P100
    plots=True,            # save curves + confusion matrix
)
SAVE_DIR = str(results.save_dir)
print('Run saved to:', SAVE_DIR)

## 7 · Validate on the held-out TEST split
Reports mAP50, mAP50-95, precision/recall — overall and per class.

In [ ]:
best = os.path.join(SAVE_DIR, 'weights', 'best.pt')
best_model = YOLO(best)
metrics = best_model.val(data=data_yaml_path, split='test', imgsz=IMGSZ, workers=WORKERS)
print('\n================ TEST METRICS ================')
print(f'mAP50-95 : {metrics.box.map:.4f}')
print(f'mAP50    : {metrics.box.map50:.4f}')
print(f'mAP75    : {metrics.box.map75:.4f}')
names = metrics.names
print('\nPer-class AP50:')
try:
    for ci, ap in zip(metrics.box.ap_class_index, metrics.box.ap50):
        print(f'  {names[int(ci)]:9s}  AP50={ap:.4f}')
except Exception as e:
    print('per-class print skipped:', e)

## 8 · Export for the website (ONNX + PyTorch)
YOLO26 is **NMS-free / end-to-end**, so the ONNX graph already outputs final decoded boxes — ideal for browser (`onnxruntime-web`) or a Python backend (`onnxruntime`).

In [ ]:
import shutil
OUT = '/kaggle/working'
# 1) PyTorch weight (use with `from ultralytics import YOLO`)
shutil.copy(best, os.path.join(OUT, 'microplastic_yolo26_best.pt'))

# 2) ONNX (portable inference)
onnx_path = None
try:
    onnx_path = best_model.export(format='onnx', imgsz=IMGSZ, opset=13, simplify=True, dynamic=False)
    shutil.copy(onnx_path, os.path.join(OUT, 'microplastic_yolo26.onnx'))
    print('ONNX exported:', onnx_path)
except Exception as e:
    print('ONNX export failed:', e)

# 3) Copy training plots for the report
for fn in ['results.png','confusion_matrix.png','confusion_matrix_normalized.png',
           'PR_curve.png','labels.jpg','args.yaml','results.csv']:
    src = os.path.join(SAVE_DIR, fn)
    if os.path.exists(src):
        shutil.copy(src, os.path.join(OUT, fn))
print('Copied artifacts to', OUT)

## 9 · Sanity-check: run inference on sample TEST images

In [ ]:
import glob, matplotlib.pyplot as plt, cv2, random
test_imgs = glob.glob(os.path.join(DATA_ROOT, 'test', 'images', '*'))
random.seed(0)
sample = random.sample(test_imgs, min(6, len(test_imgs)))
preds = best_model.predict(sample, imgsz=IMGSZ, conf=0.25, save=False)
plt.figure(figsize=(16, 10))
for i, r in enumerate(preds):
    im = cv2.cvtColor(r.plot(), cv2.COLOR_BGR2RGB)
    plt.subplot(2, 3, i+1); plt.imshow(im); plt.axis('off')
    plt.title(os.path.basename(r.path)[:22], fontsize=8)
plt.tight_layout(); plt.savefig(os.path.join(OUT,'sample_predictions.png'), dpi=120); plt.show()

## 10 · Package everything into one downloadable ZIP

In [ ]:
import shutil, os
pkg_dir = '/kaggle/working/microplastic_model_package'
os.makedirs(pkg_dir, exist_ok=True)
for f in ['microplastic_yolo26_best.pt','microplastic_yolo26.onnx','data.yaml',
          'results.png','confusion_matrix.png','PR_curve.png','results.csv',
          'sample_predictions.png','args.yaml']:
    p = os.path.join('/kaggle/working', f)
    if os.path.exists(p): shutil.copy(p, pkg_dir)
zip_path = shutil.make_archive('/kaggle/working/microplastic_model_package', 'zip', pkg_dir)
print('✅ Download this from the Output tab →', zip_path)
print('\nContents:')
for f in sorted(os.listdir(pkg_dir)):
    print('  ', f, f'({os.path.getsize(os.path.join(pkg_dir,f))/1e6:.2f} MB)')

## 11 · How to use the trained model in your website

**Option A — Python backend (FastAPI/Flask), simplest & most accurate:**
```python
from ultralytics import YOLO
model = YOLO('microplastic_yolo26_best.pt')       # or the .onnx
res = model.predict('upload.jpg', conf=0.25, imgsz=640)
for b in res[0].boxes:
    cls = res[0].names[int(b.cls)]; conf = float(b.conf)
    x1,y1,x2,y2 = b.xyxy[0].tolist()
    print(cls, round(conf,3), [round(v) for v in (x1,y1,x2,y2)])
```

**Option B — In-browser, no server (`onnxruntime-web`):** ship `microplastic_yolo26.onnx` (YOLO26 is NMS-free, so the model output is already the final boxes — you only threshold by confidence and scale coords back to the original image size). Classes are `['fiber','film','fragment','pellet']`.

**Option C — ONNX in a Python backend (no ultralytics dep):** run with `onnxruntime`, letterbox to 640×640, feed NCHW float32 `/255`, read decoded boxes from the output tensor.

---
### If the 12 h session isn't enough / it times out
1. Use **Save Version → Save & Run All (Commit)** — runs headless up to 12 h and saves `/kaggle/working`.
2. To **resume**: add this run's output as an input dataset, copy `runs` back, set `RESUME=True`, re-run. Or just lower `EPOCHS` — with early stopping (`patience=40`) the model usually converges well before 150 epochs on this dataset.